# Step 7: EEG Segmentation

Convert cleaned EEG recordings into fixed-length segments (30-second windows at 250 Hz = 7500 samples).
Only recordings that passed QC are segmented.

Key rules:
- No overlapping windows
- No padding for short recordings
- Preserve subject_id across all segments from same subject
- No train/test split yet (must be subject-level later)
- Save segments as .npy files, metadata as CSV

In [1]:
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" "mne==1.6.1" "matplotlib==3.7.2" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
anaconda-project 0.11.1 requires ruamel-yaml, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.
conda-repo-cli 1.0.20 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.20 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.
botocore 1.27.28 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
CLEANED_ROOT = PROCESSED_DIR / 'cleaned_eeg'
RESAMPLED_ROOT = PROCESSED_DIR / 'resampled'
SEG_DIR = PROCESSED_DIR / 'segmentation'
QC_DIR = PROCESSED_DIR / 'quality_control'
QC_REPORT_PATH = PROCESSED_DIR / 'quality_control_report.csv'
PREPROC_REPORT_PATH = PROCESSED_DIR / 'preprocessing_report.csv'
SAMPLING_REPORT_PATH = PROCESSED_DIR / 'sampling_rate_report.csv'
BRAINLAT_MAPPING_PATH = PROCESSED_DIR / 'brainlat_channel_mapping.csv'
SEG_META_PATH = SEG_DIR / 'segment_metadata.csv'
SEG_QC_DIR = QC_DIR / 'segmentation'

SEG_DIR.mkdir(parents=True, exist_ok=True)
(SEG_DIR / 'ADFTD').mkdir(exist_ok=True)
(SEG_DIR / 'BrainLat').mkdir(exist_ok=True)
(SEG_DIR / 'AD_Auditory').mkdir(exist_ok=True)
SEG_QC_DIR.mkdir(parents=True, exist_ok=True)
(SEG_QC_DIR / 'ADFTD').mkdir(exist_ok=True)
(SEG_QC_DIR / 'BrainLat').mkdir(exist_ok=True)
(SEG_QC_DIR / 'AD_Auditory').mkdir(exist_ok=True)

TARGET_CHANNELS = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3',
    'Pz', 'P4', 'T6', 'O1', 'O2'
]
TARGET_SFREQ = 250.0
SEG_DURATION = 30.0
SEG_SAMPLES = int(SEG_DURATION * TARGET_SFREQ)

print(f'Segment config: {SEG_DURATION}s @ {TARGET_SFREQ}Hz = {SEG_SAMPLES} samples')
print(f'Target channels: {len(TARGET_CHANNELS)}')

Segment config: 30.0s @ 250.0Hz = 7500 samples
Target channels: 19


## 1. Load QC-passed recordings

In [3]:
qc_report = pd.read_csv(QC_REPORT_PATH)
preproc_report = pd.read_csv(PREPROC_REPORT_PATH)
sampling_report = pd.read_csv(SAMPLING_REPORT_PATH)

pass_qc = qc_report[qc_report['quality_status'] == 'pass'].copy()
print(f'QC-passed recordings: {len(pass_qc)}')
print(pass_qc['dataset'].value_counts())

merged = pass_qc.merge(
    preproc_report[['dataset', 'subject_id', 'standard_label', 'original_file', 'processed_file']],
    left_on=['dataset', 'subject_id', 'standard_label', 'file_path'],
    right_on=['dataset', 'subject_id', 'standard_label', 'processed_file'],
    how='left'
)
merged = merged.merge(
    sampling_report[['dataset', 'subject_id', 'standard_label', 'eeg_file', 'resampled_file']],
    left_on=['dataset', 'subject_id', 'standard_label', 'original_file'],
    right_on=['dataset', 'subject_id', 'standard_label', 'eeg_file'],
    how='left'
)
print(f'Merged: {len(merged)}')

QC-passed recordings: 145
ADFTD          114
AD_Auditory     31
Name: dataset, dtype: int64
Merged: 259


## 2. Load BrainLat channel mapping

In [4]:
brainlat_mapping = pd.read_csv(BRAINLAT_MAPPING_PATH)
bl_map = dict(zip(brainlat_mapping['original_channel'], brainlat_mapping['standard_channel']))
print('BrainLat channel mapping:')
for k, v in bl_map.items():
    print(f'  {k} -> {v}')

BrainLat channel mapping:
  C29 -> Fp1
  C16 -> Fp2
  D7 -> F7
  D4 -> F3
  C21 -> Fz
  C4 -> F4
  C7 -> F8
  D24 -> T3
  D19 -> C3
  A1 -> Cz
  B22 -> C4
  B14 -> T4
  A10 -> T5
  A7 -> P3
  A19 -> Pz
  B4 -> P4
  B7 -> T6
  A16 -> O1
  A29 -> O2


In [5]:
def get_segment_file(dataset, subject_id, segment_idx):
    return SEG_DIR / dataset / f'{subject_id}_seg{segment_idx:06d}.npy'

def load_and_prepare_recording(row):
    dataset = row['dataset']
    subject_id = row['subject_id']
    
    if dataset == 'BrainLat':
        if pd.notna(row.get('resampled_file')) and Path(row['resampled_file']).exists():
            file_path = Path(row['resampled_file'])
        else:
            return None, 'resampled_file_missing'
    else:
        if pd.notna(row.get('processed_file')) and Path(row['processed_file']).exists():
            file_path = Path(row['processed_file'])
        else:
            return None, 'processed_file_missing'
    
    try:
        raw = mne.io.read_raw_fif(file_path, preload=True, verbose='ERROR')
        
        if not np.isclose(raw.info['sfreq'], TARGET_SFREQ, atol=1.0):
            raw.close()
            return None, f'sampling_rate_mismatch: {raw.info["sfreq"]}'
        
        if dataset == 'BrainLat':
            rename_dict = {orig: std for orig, std in bl_map.items() if orig in raw.ch_names}
            if rename_dict:
                raw.rename_channels(rename_dict)
            available_targets = [ch for ch in TARGET_CHANNELS if ch in raw.ch_names]
            if len(available_targets) < 19:
                raw.close()
                return None, f'insufficient_mapped_channels: {len(available_targets)}'
            raw.pick_channels(TARGET_CHANNELS, ordered=True)
        else:
            missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]
            if missing:
                raw.close()
                return None, f'missing_channels: {missing}'
            raw.pick_channels(TARGET_CHANNELS, ordered=True)
        
        if len(raw.ch_names) != 19:
            raw.close()
            return None, f'final_channel_count: {len(raw.ch_names)}'
        
        data = raw.get_data()
        
        if np.any(np.isnan(data)) or np.any(np.isinf(data)):
            raw.close()
            return None, 'nan_or_inf_values'
        
        raw.close()
        return data, None
    except Exception as exc:
        return None, f'{type(exc).__name__}: {str(exc)[:100]}'

## 3. Segment all recordings

In [6]:
segment_rows = []
segment_counter = 0
short_recordings = []
failed_recordings = []

for idx, row in merged.iterrows():
    dataset = row['dataset']
    subject_id = row['subject_id']
    standard_label = row['standard_label']
    
    if idx % 20 == 0:
        print(f'Processing {idx+1}/{len(merged)}: {dataset}/{subject_id}')
    
    data, error = load_and_prepare_recording(row)
    
    if error:
        failed_recordings.append({
            'dataset': dataset,
            'subject_id': subject_id,
            'standard_label': standard_label,
            'error': error
        })
        continue
    
    n_samples = data.shape[1]
    duration = n_samples / TARGET_SFREQ
    n_segments = n_samples // SEG_SAMPLES
    
    if n_segments == 0:
        short_recordings.append({
            'dataset': dataset,
            'subject_id': subject_id,
            'standard_label': standard_label,
            'duration': duration,
            'n_samples': n_samples
        })
        continue
    
    for seg_idx in range(n_segments):
        start_sample = seg_idx * SEG_SAMPLES
        end_sample = start_sample + SEG_SAMPLES
        segment_data = data[:, start_sample:end_sample]
        
        if segment_data.shape != (19, SEG_SAMPLES):
            continue
        
        segment_counter += 1
        seg_file = get_segment_file(dataset, subject_id, segment_counter)
        np.save(seg_file, segment_data.astype(np.float32))
        
        segment_rows.append({
            'segment_id': segment_counter,
            'dataset': dataset,
            'subject_id': subject_id,
            'standard_label': standard_label,
            'source_file': row.get('eeg_file', row.get('original_file', '')),
            'start_time': start_sample / TARGET_SFREQ,
            'end_time': end_sample / TARGET_SFREQ,
            'sampling_rate': TARGET_SFREQ,
            'channels': ';'.join(TARGET_CHANNELS),
            'num_samples': SEG_SAMPLES
        })

segment_meta = pd.DataFrame(segment_rows)
print(f'\nTotal segments created: {len(segment_meta)}')
print(f'Short recordings (<30s): {len(short_recordings)}')
print(f'Failed recordings: {len(failed_recordings)}')

if short_recordings:
    print('\nShort recordings:')
    for r in short_recordings:
        print(f'  {r["dataset"]}/{r["subject_id"]}: {r["duration"]:.1f}s')

if failed_recordings:
    print('\nFailed recordings:')
    for r in failed_recordings[:10]:
        print(f'  {r["dataset"]}/{r["subject_id"]}: {r["error"]}')

Processing 1/259: ADFTD/sub-001
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pi

## 4. Save segment metadata and verify

In [7]:
segment_meta.to_csv(SEG_META_PATH, index=False)
print(f'Saved metadata: {SEG_META_PATH}')

verify_errors = []
for idx, row in segment_meta.iterrows():
    seg_file = SEG_DIR / row['dataset'] / f'{row["subject_id"]}_seg{row["segment_id"]:06d}.npy'
    if not seg_file.exists():
        verify_errors.append(f'missing_file: {seg_file}')
        continue
    try:
        seg_data = np.load(seg_file)
        if seg_data.shape != (19, SEG_SAMPLES):
            verify_errors.append(f'shape_mismatch {row["segment_id"]}: {seg_data.shape}')
        if np.any(np.isnan(seg_data)) or np.any(np.isinf(seg_data)):
            verify_errors.append(f'nan_inf {row["segment_id"]}')
    except Exception as exc:
        verify_errors.append(f'load_error {row["segment_id"]}: {exc}')

print(f'Verification errors: {len(verify_errors)}')
if verify_errors:
    for e in verify_errors[:10]:
        print(f'  {e}')
else:
    print('All segments verified OK')

Saved metadata: /home/admin1/Desktop/SIH_2026_ALL/processed_data/segmentation/segment_metadata.csv
Verification errors: 0
All segments verified OK


## 5. Summary tables

In [8]:
ds_summary = segment_meta.groupby('dataset').agg(
    Subjects=('subject_id', 'nunique'),
    Recordings=('source_file', 'nunique'),
    Segments=('segment_id', 'count')
).reset_index()
print('DATASET SUMMARY:')
print(ds_summary.to_string(index=False))

label_summary = segment_meta.groupby('standard_label').agg(
    Subjects=('subject_id', 'nunique'),
    Segments=('segment_id', 'count')
).reset_index()
print('\nLABEL SUMMARY:')
print(label_summary.to_string(index=False))

subj_summary = segment_meta.groupby(['subject_id', 'dataset', 'standard_label']).size().reset_index(name='Number_of_Segments')
subj_summary = subj_summary.sort_values(['dataset', 'subject_id']).reset_index(drop=True)
print(f'\nSUBJECT SUMMARY ({len(subj_summary)} subjects):')
print(subj_summary.to_string(index=False))

DATASET SUMMARY:
    dataset  Subjects  Recordings  Segments
      ADFTD        57         114      5848
AD_Auditory        31          31       533

LABEL SUMMARY:
standard_label  Subjects  Segments
            AD        36      2683
           FTD        17      1536
            HC        27      2018
           MCI         6       114
       UNKNOWN         2        30

SUBJECT SUMMARY (88 subjects):
subject_id     dataset standard_label  Number_of_Segments
   sub-001       ADFTD             AD                  76
   sub-002       ADFTD             AD                 104
   sub-003       ADFTD             AD                  40
   sub-004       ADFTD             AD                  92
   sub-005       ADFTD             AD                 104
   sub-006       ADFTD             AD                  84
   sub-007       ADFTD             AD                 100
   sub-009       ADFTD             AD                  80
   sub-010       ADFTD             AD                 172
   sub-015   

## 6. Visualization: representative segments

In [9]:
def plot_segment(dataset, subject_id, segment_id, seg_data, save_dir):
    try:
        fig, ax = plt.subplots(figsize=(16, 10))
        times = np.arange(SEG_SAMPLES) / TARGET_SFREQ
        
        for i, ch in enumerate(TARGET_CHANNELS):
            offset = i * 100
            ax.plot(times, seg_data[i] * 1e6 + offset, label=ch, linewidth=0.5)
        
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Amplitude (µV, offset per channel)')
        ax.set_title(f'{dataset} | {subject_id} | Segment {segment_id} | {SEG_DURATION}s @ {TARGET_SFREQ}Hz')
        ax.set_xlim(0, SEG_DURATION)
        ax.legend(ncol=4, fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        save_path = save_dir / dataset / f'{subject_id}_seg{segment_id:06d}_plot.png'
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
    except Exception as exc:
        print(f'Plot failed for {dataset}/{subject_id}: {exc}')

for dataset in ['ADFTD', 'BrainLat', 'AD_Auditory']:
    subset = segment_meta[segment_meta['dataset'] == dataset].head(3)
    for _, row in subset.iterrows():
        seg_file = SEG_DIR / dataset / f'{row["subject_id"]}_seg{row["segment_id"]:06d}.npy'
        if seg_file.exists():
            seg_data = np.load(seg_file)
            plot_segment(dataset, row['subject_id'], row['segment_id'], seg_data, SEG_QC_DIR)
    print(f'Segmentation plots generated for {dataset}')

Segmentation plots generated for ADFTD
Segmentation plots generated for BrainLat
Segmentation plots generated for AD_Auditory


## 7. Final check and report

In [10]:
print('='*60)
print('FINAL SEGMENTATION REPORT')
print('='*60)

total_subjects = segment_meta['subject_id'].nunique()
total_recordings = segment_meta['source_file'].nunique()
total_segments = len(segment_meta)

ad_subjects = segment_meta[segment_meta['standard_label'] == 'AD']['subject_id'].nunique()
ftd_subjects = segment_meta[segment_meta['standard_label'] == 'FTD']['subject_id'].nunique()
hc_subjects = segment_meta[segment_meta['standard_label'] == 'HC']['subject_id'].nunique()
mci_subjects = segment_meta[segment_meta['standard_label'] == 'MCI']['subject_id'].nunique()
unknown_subjects = segment_meta[segment_meta['standard_label'] == 'UNKNOWN']['subject_id'].nunique()

print(f'Total subjects: {total_subjects}')
print(f'Total recordings: {total_recordings}')
print(f'Total segments: {total_segments}')
print(f'AD subjects: {ad_subjects}')
print(f'FTD subjects: {ftd_subjects}')
print(f'HC subjects: {hc_subjects}')
print(f'MCI subjects: {mci_subjects}')
print(f'UNKNOWN subjects: {unknown_subjects}')

passed_subjects = set(pass_qc['subject_id'].unique())
segmented_subjects = set(segment_meta['subject_id'].unique())
zero_segment_subjects = passed_subjects - segmented_subjects
print(f'\nSubjects with zero usable segments: {len(zero_segment_subjects)}')
for s in sorted(zero_segment_subjects):
    print(f'  {s}')

print(f'\nSegment files saved to: {SEG_DIR}')
for ds in ['ADFTD', 'BrainLat', 'AD_Auditory']:
    n = len(list((SEG_DIR / ds).glob('*.npy')))
    print(f'  {ds}: {n} .npy files')

print('\nEEG SEGMENTATION COMPLETE — READY FOR COMBINED DATASET CREATION')

FINAL SEGMENTATION REPORT
Total subjects: 88
Total recordings: 145
Total segments: 6381
AD subjects: 36
FTD subjects: 17
HC subjects: 27
MCI subjects: 6
UNKNOWN subjects: 2

Subjects with zero usable segments: 0

Segment files saved to: /home/admin1/Desktop/SIH_2026_ALL/processed_data/segmentation
  ADFTD: 5848 .npy files
  BrainLat: 0 .npy files
  AD_Auditory: 533 .npy files

EEG SEGMENTATION COMPLETE — READY FOR COMBINED DATASET CREATION
